In [1]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.50"
import jax.numpy as jnp

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [2]:
data_config = _config.get_config('pi05_libero_skill_reason_fixed')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

The dataset you requested (None) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=None
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

Using new skill reasoning dataset


In [3]:
import os
from pathlib import Path
import sys
SCRIPT_DIR = Path("../py_script")
sys.path.append(str(SCRIPT_DIR))
from vlm_interfaces import *

In [18]:
import importlib
import vla_verify
importlib.reload(vla_verify.scene_graph)
from vla_verify.scene_graph import TaskSceneGraph
from vla_verify.verifier import VLAVerifier
PDDL_PATH = SCRIPT_DIR / "pddl" / "libero_domain.pddl"
pddl_domain_text = open(PDDL_PATH).read()

llm_interface, vlm_interface = get_openrouter_interfaces()
scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)
verifier = VLAVerifier(scene_graph, llm_interface)

Using OpenRouter
  LLM: google/gemini-3-flash-preview
  VLM: google/gemini-3.1-pro-preview


In [9]:
def image_tensor_to_cv2(image, resolution=(512,512)):
    return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)

def get_episode(episode_idx):
    reasonings = dataset.reasoning[episode_idx]
    start_idx = dataset.episode_starts[episode_idx]
    end_idx = dataset.episode_ends[episode_idx]
    data = dataset.hf_dataset[int(start_idx)]
    video_frames = []
    for i in range(start_idx, end_idx):
        img_data = dataset.hf_dataset[i]['image']
        video_frames.append(image_tensor_to_cv2(img_data))
    return reasonings, video_frames

print(len(dataset.episode_starts))
reasonings, video_frames = get_episode(391)
print(reasonings['instruction'])
mediapy.write_video(f'sample.mp4', video_frames, fps=20)

4338


/tmp/ipykernel_2161734/4137807961.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)


put the butter at the front in the top drawer of the cabinet and close it


In [19]:
out_dir = "data"
import gzip
import os
import re
import json
import scipy
os.makedirs(out_dir, exist_ok=True)

def process_episode(episode, episode_idx, subsample=5):
    reasonings, video_frames = episode
    task = reasonings['instruction'].split(':', 1)[-1].strip()
    print(task)
    subsample_frames = video_frames[::subsample]
    scene_graph.read_image(subsample_frames, hint=f"The robot is trying to {task}", ground=True)
    video_results = scene_graph.ground_video(additional_points_labels=[
        ("robot", [[255, 90]])
    ])

    targets = []
    target_names_and_valid = []
    for segment in reasonings['segments']:
        skill = segment['skill']
        splits = re.split('([^a-zA-Z0-9]left[^a-zA-Z0-9]|[^a-zA-Z0-9]right[^a-zA-Z0-9])', skill)
        for i in range(1, len(splits), 2):
            if len(splits[i]) == 6:
                splits[i] = splits[i][0] + 'right' + splits[i][-1]
            elif len(splits[i]) == 7:
                splits[i] = splits[i][0] + 'left' + splits[i][-1]
        skill = ''.join(splits)
        skill_res = verifier.verify_skill(skill)
        print(skill_res)
        if not skill_res.accepted:
            print("Verification failed!")
            break
        name, params = verifier._normalize_grounded_action(skill_res.grounded_action)
        pddl_action = scene_graph.match_grounded_action(name, params)
        if pddl_action is None:
            print("Could not match pddl action!")
            break
        target = None
        match pddl_action.name.value:
            case "pickup_from" | "open" | "place_in" | "turn_on" | "turn_off":
                target = pddl_action.grounding[0].value
            case "place_on":
                target = pddl_action.grounding[2].value
                # TODO: table location is bad
        start_step = segment['start_step']
        end_step = segment['end_step']
        
        valid = False
        if target is not None and target in video_results[0]:
            valid = True
            frame_start = start_step // subsample
            frame_end = end_step // subsample
            # Pad with an extra frame for interpolation, if we are not at end of sequence
            if frame_end <= frame_start:
                frame_end = frame_start + 1
            if frame_end < len(video_results) - 1:
                frame_end += 1

            def box_midpoint(box):
                return (box[0]+box[2]/2, box[1]+box[3]/2)
            positions = []
            frame_times = []
            all_frame_times = np.arange(frame_start, frame_end) * 5
            for t, r in zip(all_frame_times, video_results[frame_start:frame_end]):
                if target in r:
                    positions.append(box_midpoint(r[target].box_xywh))
                    frame_times.append(t)
    
            if len(positions) > 0:
                full_times = np.array(list(range(start_step, end_step)))
                positions_interp = scipy.interpolate.interp1d(frame_times, positions, axis=0, bounds_error=False, fill_value=(positions[0], positions[-1]))(full_times)
                targets.extend(positions_interp)
            else:
                valid = False
        if not valid:
            targets.extend([[0, 0]]*(end_step - start_step))
        target_names_and_valid.extend([[target, valid]] * (end_step - start_step))
        scene_graph.apply_action(pddl_action)

    # NOTE: This may not contain the full trace, if verification fails...
    with gzip.open(f'{out_dir}/{episode_idx}_targets.json.zip', 'wt', encoding="ascii") as zipfile:
        json.dump(target_names_and_valid, zipfile)
    np.save(f'{out_dir}/{episode_idx}_targets.npy', np.array(targets))

In [21]:
%env CC=/usr/bin/gcc
results = process_episode((reasonings, video_frames), 391)

env: CC=/usr/bin/gcc
put the butter at the front in the top drawer of the cabinet and close it
  [VLM] Querying VLM for scene graph construction...


INFO 2026-04-02 21:15:43,096 2161734 sam3_video_predictor.py: 302: using the following GPU IDs: [0]
INFO 2026-04-02 21:15:43,097 2161734 sam3_video_predictor.py: 318: 


	*** START loading model on all ranks ***


INFO 2026-04-02 21:15:43,097 2161734 sam3_video_predictor.py: 320: loading model on rank=0 with world_size=1 -- this could take a while ...


  [VLM] Time elapsed: 20.573272404726595
raw_pddl_state (define (problem tabletop_problem)
 (:domain tabletop)
 (:objects
  robot_0 - robot
  table_0 - immovable ; a light brown wooden table | in the center of the scene
  cabinet_0 - immovable ; a dark brown cabinet | on the right side of the table
  cabinet_0_top_drawer - immovable ; the top drawer of the dark brown cabinet | on the right side of the table
  cabinet_0_bottom_drawer - immovable ; the bottom drawer of the dark brown cabinet | on the right side of the table
  bowl_0 - movable ; a white bowl | in the middle of the table
  butter_0 - movable ; a stick of butter in orange packaging | at the front left of the table
  butter_1 - movable ; a stick of butter in orange packaging | at the back left of the table
  butter_2 - movable ; a stick of butter in orange packaging | in the middle of the table
 )
 (:init
  (free robot_0)
  (on cabinet_0 table_0)
  (part cabinet_0_top_drawer cabinet_0)
  (part cabinet_0_bottom_drawer cabinet

INFO 2026-04-02 21:16:01,157 2161734 sam3_video_base.py: 125: setting max_num_objects=10000 and num_obj_for_compile=16
INFO 2026-04-02 21:16:01,879 2161734 sam3_video_predictor.py: 322: loading model on rank=0 with world_size=1 -- DONE locally
INFO 2026-04-02 21:16:01,880 2161734 sam3_video_predictor.py: 333: 


	*** DONE loading model on all ranks ***




Grounding objects:
0 a a light brown wooden table in the center of the scene
1 a a dark brown cabinet on the right side of the table
2 a the top drawer of the dark brown cabinet on the right side of the table
3 a the bottom drawer of the dark brown cabinet on the right side of the table
4 a a white bowl in the middle of the table
5 a a stick of butter in orange packaging at the front left of the table
6 a a stick of butter in orange packaging at the back left of the table
7 a a stick of butter in orange packaging in the middle of the table
14 tracks.
8 objects.
Running CLIP...
Matching: {6: 'stick_of_butter_in_orange_packaging_at_the_front_left_of_the_table_2', 5: 'stick_of_butter_in_orange_packaging_at_the_front_left_of_the_table_0', 4: 'light_brown_wooden_table_in_the_center_of_the_scene_0', 3: 'the_top_drawer_of_the_dark_brown_cabinet_on_the_right_side_of_the_table_9', 7: 'the_top_drawer_of_the_dark_brown_cabinet_on_the_right_side_of_the_table_1', 2: 'the_top_drawer_of_the_dark_brow

...

propagate_in_video: 0it [00:00, ?it/s]

Done.
Propagating detections...

  0%|          | 0/37 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Done.
  [LLM] Querying LLM for subtask translation...
  [LLM] Time elapsed: 182.21246405830607
SubtaskVerificationResult(subtask='PICKUP_FROM(front butter, table)', accepted=True, reasoning="The skill specifies 'front butter' on the 'table'. In the scene graph, butter_0 is described as being 'at the front left of the table', butter_1 is 'at the back left', and butter_2 is 'in the middle'. Therefore, butter_0 is the best match for 'front butter'.", grounded_action={'name': 'pickup_from', 'parameters': ['butter_0', 'robot_0', 'table_0']})
  [LLM] Querying LLM for subtask translation...
  [LLM] Time elapsed: 1.7365832859650254
SubtaskVerificationResult(subtask='PLACE_IN(front butter, top drawer)', accepted=True, reasoning="The skill 'PLACE_IN(front butter, top drawer)' matches the robot carrying 'butter_0' (located at the front left of the table) and placing it into 'cabinet_0_top_drawer' (the top drawer). The top drawer is currently 'open', which satisfies the PDDL precondition for the '

In [92]:
print(reasonings['segments'])

In [80]:
import re
import scipy
targets = []
target_names = []
scene_graph.reset_simulator()
subsample = 5
for segment in reasonings['segments']:
    skill = segment['skill']
    splits = re.split('([^a-zA-Z0-9]left[^a-zA-Z0-9]|[^a-zA-Z0-9]right[^a-zA-Z0-9])', skill)
    for i in range(1, len(splits), 2):
        if len(splits[i]) == 6:
            splits[i] = splits[i][0] + 'right' + splits[i][-1]
        elif len(splits[i]) == 7:
            splits[i] = splits[i][0] + 'left' + splits[i][-1]
    skill = ''.join(splits)
    skill_res = verifier.verify_skill(skill)
    print(skill_res)
    if not skill_res.accepted:
        print("Verification failed!")
        break
    name, params = verifier._normalize_grounded_action(skill_res.grounded_action)
    pddl_action = scene_graph.match_grounded_action(name, params)
    if pddl_action is None:
        print("Could not match pddl action!")
        break
    target = None
    match pddl_action.name.value:
        case "pickup_from" | "open" | "place_in" | "turn_on" | "turn_off":
            target = pddl_action.grounding[0].value
        case "place_on":
            target = pddl_action.grounding[2].value
            # TODO: table location is bad
    start_step = segment['start_step']
    end_step = segment['end_step']
    if target is not None:
        frame_start = start_step // subsample
        frame_end = end_step // subsample
        # Pad with an extra frame for interpolation, if we are not at end of sequence

        if frame_end <= frame_start:
            frame_end = frame_start + 1

        if frame_end < len(results) - 1:
            frame_end += 1
        def box_midpoint(box):
            return (box[0]+box[2]/2, box[1]+box[3]/2)
        frame_times = np.array(list(range(frame_start, frame_end))) * 5
        positions = np.array([box_midpoint(r[target].box_xywh) for r in results[frame_start:frame_end]])

        full_times = np.array(list(range(start_step, end_step)))
        positions_interp = scipy.interpolate.interp1d(frame_times, positions, axis=0, bounds_error=False, fill_value=(positions[0], positions[-1]))(full_times)
        targets.extend(positions_interp)
    else:
        targets.extend([None]*(end_step - start_step))
    target_names.extend([target] * (end_step - start_step))
        
    scene_graph.apply_action(pddl_action)


In [84]:
plot_targets = np.array(targets)[:180]
plt.scatter(plot_targets[:, 0], -plot_targets[:, 1])
plt.xlim((0, 1))
plt.ylim((-1, 0))
plt.gca().set_aspect('equal')
plt.show()